# T7A — LLM — format / tone / template

**Lemma D7** · `nuisance="style"` · [Task doc](../../docs/tasks/t07a-llm-style.md) · FINAL: `paper_code/T7/task7A/FINAL.md`

> Matched $\Sigma_{\text{style}}$ RM: sycophancy **38.5%→13.5%**, style gap **2.199→0.803**; margin_pmh DPO Style TDI **1.836**.

| § | What you do |
|---|-------------|
| 1–4 | Install → load demo → `check_applicability` |
| 5–6 | Estimate $\Sigma_{\text{task}}$ → PMH train → Step 5 on deploy holdout |
| 7–8 | Reproduce paper scripts → plug in your data |


## 1 — Install


In [ ]:
!pip install -q "matching-pmh[hf]"


## 2 — Config & imports


In [ ]:
import os
import torch
from pmh.benchmark.presets import get_preset
from pmh import check_applicability
from pmh.adoption import RECIPE_ONE_LINER

QUICK = os.environ.get("PMH_QUICK", "").lower() in ("1", "true", "yes")
SEED = 0
print(RECIPE_ONE_LINER)


## 3 — Load demo data


In [ ]:
preset = get_preset("t7a_style_d7")
import json, tempfile
from pathlib import Path

import torch
from pmh.integrations.huggingface import load_style_pairs_jsonl

rows = [
    {
        "id": "ex1",
        "prompt": "Summarize the paper.",
        "content_fixed": "The method matches deployment nuisance covariance.",
        "style_variants": {
            "bulleted": "- Matches Sigma_task\n- Adds PMH penalty",
            "verbose": "In this detailed response, we explain matching at length.",
        },
    },
]
path = Path(tempfile.mkstemp(suffix=".jsonl")[1])
with path.open("w", encoding="utf-8") as f:
    for row in rows:
        f.write(json.dumps(row) + "\n")

pairs = load_style_pairs_jsonl(path)

class HashEncoder(torch.nn.Module):
    def __init__(self, dim: int = 64) -> None:
        super().__init__()
        self.dim = dim
        self.dummy = torch.nn.Parameter(torch.zeros(1))

    def forward(self, input_ids, attention_mask=None, **kwargs):
        del attention_mask, kwargs
        b, t = input_ids.shape
        h = torch.zeros(b, t, self.dim)
        for i in range(b):
            h[i] = torch.nn.functional.one_hot(input_ids[i] % self.dim, self.dim).float().mean(0)
        return type("Out", (), {"hidden_states": (h,)})()

class ToyTokenizer:
    pad_token = "<pad>"
    eos_token = "</s>"
    chat_template = None

    def __call__(self, texts, return_tensors="pt", padding=True, truncation=True, max_length=128):
        rows = [[hash(w) % 997 for w in t.split()[:max_length]] or [0] for t in texts]
        max_len = max(len(r) for r in rows)
        input_ids = torch.zeros(len(rows), max_len, dtype=torch.long)
        mask = torch.zeros(len(rows), max_len, dtype=torch.long)
        for i, r in enumerate(rows):
            input_ids[i, : len(r)] = torch.tensor(r, dtype=torch.long)
            mask[i, : len(r)] = 1
        return {"input_ids": input_ids, "attention_mask": mask}

encoder = HashEncoder(64)
tokenizer = ToyTokenizer()
print(len(pairs), "style pairs loaded")


## 4 — Scope (applicability)


In [ ]:
from pmh import check_applicability

app = check_applicability(stack="hf", has_style_pairs=True)
print(app.summary())


## 5 — Estimate $\Sigma_{\text{task}}$ + PMH train


In [ ]:
from pmh.integrations.huggingface import estimate_style_sigma

rank = preset.default_rank if preset else 8
artifact = estimate_style_sigma(pairs, encoder, tokenizer, rank=rank, batch_size=4)
print("preflight", artifact.preflight, "trace", artifact.sigma.trace().item())


## 6 — Step 5 (deploy holdout)


In [ ]:
# Step 5 for HF: attach artifact to your causal LM, then run deployment eval.
# PMHTrainer.from_artifact(model, hook=last_hidden_state_layer, artifact=artifact, pmh_config=preset.pmh_config)
print("artifact ready — plug into your HF training loop (see task doc §8)")


## 7 — Paper reproduction


Frozen results: `paper_code/T7/task7A/FINAL.md`

- **RM behavioral eval (TQA n=500):** `python paper_code/T7/task7A/evaluate_7a_behavioral.py`
- **Geometric DPO + style geometry:** `python paper_code/T7/task7A/train_geometric_dpo.py`
- **Synthetic alignment pipeline:** `python paper_code/T7/task7A/run_task7a_pipeline.py`


## 8 — Your pipeline


Style-pair JSONL (same content, two surfaces) → `estimate_style_sigma` / D7 trainer.
